In [1]:
import numpy as np
import pandas as pd
import scipy
import torch
import time
import platform

from scipy.sparse import diags
from IPython.display import display

print("Environment initialized successfully!")

Environment initialized successfully!


In [2]:
def generate_spd_matrix(n):

    diagonals = [
        -np.ones(n - 1),
         2 * np.ones(n),
        -np.ones(n - 1)
    ]

    A = diags(
        diagonals,
        offsets=[-1, 0, 1],
        format="csr"
    )

    return A

In [3]:
matrix_sizes = [1000, 5000, 10000, 20000, 50000]

matrix_info = []

for n in matrix_sizes:

    A = generate_spd_matrix(n)

    nnz = A.nnz
    total_elements = n * n
    sparsity = 1 - (nnz / total_elements)

    matrix_info.append({
        "Matrix_Size": n,
        "Non_Zero_Elements": nnz,
        "Sparsity_Percentage": sparsity * 100,
        "Data_Type": str(A.dtype)
    })

matrix_df = pd.DataFrame(matrix_info)

display(matrix_df)

,Matrix_Size,Non_Zero_Elements,Sparsity_Percentage,Data_Type
0,1000,2998,99.700200,float64
1,5000,14998,99.940008,float64
2,10000,29998,99.970002,float64
3,20000,59998,99.985000,float64
4,50000,149998,99.994000,float64


In [4]:
def generate_cg_problem(n):

    A = generate_spd_matrix(n)

    x_true = np.ones(n, dtype=np.float64)

    b = A @ x_true

    x0 = np.zeros(n, dtype=np.float64)

    return A, b, x0, x_true

In [12]:

import numpy as np
from scipy.sparse import diags
from numba import cuda

N = 1000

main_diags = 2.0 * np.ones(N, dtype=np.float64)
off_diags = -1.0 * np.ones(N-1, dtype=np.float64)

A = diags(
    [off_diags, main_diags, off_diags],
    offsets=[-1, 0, 1],
    format='csr'
)

# Transfer CSR components to GPU
values = cuda.to_device(A.data.astype(np.float64))
col_indices = cuda.to_device(A.indices.astype(np.int32))
row_ptr = cuda.to_device(A.indptr.astype(np.int32))

# Generate known solution and RHS
x_true = np.ones(N, dtype=np.float64)
b = A @ x_true

print("Matrix size:", A.shape)
print("Non-zero elements:", A.nnz)
print("Data type:", values.dtype)
print("GPU transfer successful:", values.size == A.nnz)
print("RHS shape:", b.shape)


Matrix size: (1000, 1000)
Non-zero elements: 2998
Data type: float64
GPU transfer successful: True
RHS shape: (1000,)


In [7]:
from numba import cuda
import numpy as np

@cuda.jit
def csr_spmv(values, col_indices, row_ptr, x, y, n):
    row = cuda.grid(1)

    if row < n:
        row_sum = 0.0

        for j in range(row_ptr[row], row_ptr[row + 1]):
            row_sum += values[j] * x[col_indices[j]]

        y[row] = row_sum


N = A.shape[0]

x = np.ones(N, dtype=np.float32)
y = np.zeros(N, dtype=np.float32)

# Transfer vectors to GPU
d_x = cuda.to_device(x)
d_y = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256

blocks_per_grid = (
    N + threads_per_block - 1
) // threads_per_block

csr_spmv[blocks_per_grid, threads_per_block](
    values,
    col_indices,
    row_ptr,
    d_x,
    d_y,
    N
)

# Copy output from GPU
y_gpu = d_y.copy_to_host()

print("First 10 GPU results:", y_gpu[:10])


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


First 10 GPU results: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [8]:

@cuda.jit
def dot_product_kernel(a, b, partial, n):

    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    shared = cuda.shared.array(256, dtype=np.float64)

    value = 0.0

    if idx < n:
        value = a[idx] * b[idx]

    shared[tid] = value
    cuda.syncthreads()

    stride = cuda.blockDim.x // 2

    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]

        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        partial[cuda.blockIdx.x] = shared[0]


N = 1000

a = np.ones(N, dtype=np.float32)
b = np.ones(N, dtype=np.float32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)

threads_per_block = 256

blocks_per_grid = (
    N + threads_per_block - 1
) // threads_per_block

d_partial = cuda.device_array(
    blocks_per_grid,
    dtype=np.float32
)


dot_product_kernel[blocks_per_grid, threads_per_block](
    d_a,
    d_b,
    d_partial,
    N
)

# Retrieve partial sums
partial_results = d_partial.copy_to_host()

# Final reduction on CPU
gpu_dot = np.sum(partial_results, dtype=np.float64)

# CPU reference
cpu_dot = np.dot(a, b)

print("GPU Dot Product:", gpu_dot)
print("CPU Dot Product:", cpu_dot)

print("Validation:", np.isclose(gpu_dot, cpu_dot))

GPU Dot Product: 1000.0
CPU Dot Product: 1000.0
Validation: True


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [9]:

@cuda.jit
def vector_update(a, b, result, alpha, operation, n):

    i = cuda.grid(1)

    if i < n:

        if operation == 0:
            # result = a + alpha * b
            result[i] = a[i] + alpha * b[i]

        elif operation == 1:
            # result = a - alpha * b
            result[i] = a[i] - alpha * b[i]

        elif operation == 2:
            # result = alpha * a
            result[i] = alpha * a[i]


N = 1000

a = np.ones(N, dtype=np.float32) * 10
b = np.ones(N, dtype=np.float32) * 2

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)

d_result = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

vector_update[blocks_per_grid, threads_per_block](
    d_a, d_b, d_result, 2.0, 0, N
)

result = d_result.copy_to_host()

print("Addition:", result[:5])

vector_update[blocks_per_grid, threads_per_block](
    d_a, d_b, d_result, 2.0, 1, N
)

result = d_result.copy_to_host()

print("Subtraction:", result[:5])

vector_update[blocks_per_grid, threads_per_block](
    d_a, d_b, d_result, 2.0, 2, N
)

result = d_result.copy_to_host()

print("Scalar multiplication:", result[:5])


Addition: [14. 14. 14. 14. 14.]
Subtraction: [6. 6. 6. 6. 6.]
Scalar multiplication: [20. 20. 20. 20. 20.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [10]:
import numpy as np
from numba import cuda

def gpu_conjugate_gradient(
    values,
    col_indices,
    row_ptr,
    b,
    tol=1e-6,
    max_iter=10000
):
    n = len(b)

    # Transfer input data to GPU
    dtype=np.float64

    d_b = cuda.to_device(b.astype(dtype))
    d_x = cuda.to_device(np.zeros(n, dtype=dtype))

    d_r = cuda.device_array(n, dtype=dtype)
    d_p = cuda.device_array(n, dtype=dtype)
    d_Ap = cuda.device_array(n, dtype=dtype)

    threads = 256
    blocks = (n + threads - 1) // threads

    d_partial = cuda.device_array(
        blocks, dtype=dtype
    )

    # Initial values: x = 0, r = b, p = r
    d_r.copy_to_device(d_b)
    d_p.copy_to_device(d_b)

    # Initial residual squared
    dot_product_kernel[blocks, threads](
        d_r, d_r, d_partial, n
    )
    rs_old = np.sum(d_partial.copy_to_host(), dtype=np.float64)

    initial_norm = np.sqrt(rs_old)

    if initial_norm == 0:
        return np.zeros(n, dtype=np.float32), 0, 0.0

    for iteration in range(max_iter):

        # Ap = A * p
        csr_spmv[blocks, threads](
            values, col_indices, row_ptr,
            d_p, d_Ap, n
        )

        # pAp = p.T * Ap
        dot_product_kernel[blocks, threads](
            d_p, d_Ap, d_partial, n
        )
        pAp = np.sum(
            d_partial.copy_to_host(), dtype=np.float64
        )

        if pAp <= 0:
            raise ValueError("Matrix may not be positive definite.")

        alpha = rs_old / pAp

        # x = x + alpha * p
        vector_update[blocks, threads](
            d_x, d_p, d_x, alpha, 0, n
        )

        # r = r - alpha * Ap
        vector_update[blocks, threads](
            d_r, d_Ap, d_r, alpha, 1, n
        )

        # Calculate new residual squared
        dot_product_kernel[blocks, threads](
            d_r, d_r, d_partial, n
        )
        rs_new = np.sum(
            d_partial.copy_to_host(), dtype=np.float64
        )

        residual_norm = np.sqrt(rs_new)

        if residual_norm <= tol * initial_norm:

              # Recalculate the true residual: b - A*x
              csr_spmv[blocks, threads](
                  values,
                  col_indices,
                  row_ptr,
                  d_x,
                  d_Ap,
                  n
              )

              Ax_host = d_Ap.copy_to_host()

              true_r = b - Ax_host

              true_norm = np.linalg.norm(true_r)

              # Check actual convergence
              if true_norm <= tol * initial_norm:
                  return (
                      d_x.copy_to_host(),
                      iteration + 1,
                      true_norm
                  )

              # Residual replacement
              d_r.copy_to_device(true_r)
              d_p.copy_to_device(true_r)

              # Recalculate residual squared
              dot_product_kernel[blocks, threads](
                  d_r, d_r, d_partial, n
              )

              rs_old = np.sum(
                  d_partial.copy_to_host(),
                  dtype=np.float64
              )

              # Restart CG using the corrected residual
              continue

        beta = rs_new / rs_old

        # p = r + beta * p
        vector_update[blocks, threads](
            d_r, d_p, d_p, beta, 0, n
        )

        rs_old = rs_new

    return d_x.copy_to_host(), max_iter, np.sqrt(rs_old)

In [11]:
import time

def benchmark_gpu_cg(n, tol=1e-6, max_iter=100000):
    A, b, x0, x_true = generate_cg_problem(n)

    d_values = cuda.to_device(A.data.astype(np.float64))
    d_col_indices = cuda.to_device(A.indices.astype(np.int32))
    d_row_ptr = cuda.to_device(A.indptr.astype(np.int32))

    # Warm up CUDA kernels and JIT compilation
    if not hasattr(benchmark_gpu_cg, "_warmed_up"):
        gpu_conjugate_gradient(
            d_values, d_col_indices, d_row_ptr,
            b, tol=tol, max_iter=max_iter
        )
        benchmark_gpu_cg._warmed_up = True

    cuda.synchronize()
    start = time.perf_counter()

    x_gpu, iterations, returned_residual = gpu_conjugate_gradient(
        d_values,
        d_col_indices,
        d_row_ptr,
        b,
        tol=tol,
        max_iter=max_iter
    )

    cuda.synchronize()
    execution_time = (time.perf_counter() - start) * 1000

    relative_residual = (
        np.linalg.norm(A @ x_gpu - b) / np.linalg.norm(b)
    )

    return {
        "Matrix_Size": n,
        "NNZ": A.nnz,
        "Hardware": "Tesla T4",
        "Execution_Time_ms": execution_time,
        "Iterations": iterations,
        "Relative_Residual": relative_residual,
        "Converged": relative_residual <= tol,
        "Solution_Error": np.linalg.norm(x_gpu - x_true)
    }

In [14]:

# Execute GPU Conjugate Gradient solver

tol = 1e-6
max_iter = 2000

start = time.perf_counter()

x_gpu, iterations, residual = gpu_conjugate_gradient(
    values,
    col_indices,
    row_ptr,
    b,
    tol,
    max_iter
)

cuda.synchronize()

execution_time = (time.perf_counter() - start) * 1000

# Validate against the true solution
true_residual = np.linalg.norm(A @ x_gpu - b) / np.linalg.norm(b)
solution_error = np.linalg.norm(x_gpu - x_true)

print("GPU CG Validation")
print("-------------------------")
print("Matrix size:", N)
print("Iterations:", iterations)
print("Returned residual:", residual)
print("True relative residual:", true_residual)
print("Solution error:", solution_error)
print("Execution time:", execution_time, "ms")
print("Converged:", true_residual < tol)


GPU CG Validation
-------------------------
Matrix size: 1000
Iterations: 500
Returned residual: 5.189118401918117e-13
True relative residual: 3.669260810376201e-13
Solution error: 1.1320862088232394e-12
Execution time: 1079.9393109999755 ms
Converged: True


# CG Application

In [15]:
import gradio as gr
import os

# -----------------------------
# Configuration
# -----------------------------

TOL = 1e-6

MATRIX_SIZES = [1000, 5000, 10000, 20000, 50000]

RESULT_FILE = "/content/week09_cloud_cg_results.csv"

# -----------------------------
# Verify cloud GPU
# -----------------------------

if not cuda.is_available():
    raise RuntimeError("CUDA GPU is not available!")

print("Cloud GPU:", cuda.get_current_device().name)
print("CUDA available:", cuda.is_available())

# -----------------------------
# CG execution function
# -----------------------------

def execute_cg(n):

    # Generate sparse SPD matrix
    main_diag = 2.0 * np.ones(n, dtype=np.float64)
    off_diag = -1.0 * np.ones(n - 1, dtype=np.float64)

    A = diags(
        [off_diag, main_diag, off_diag],
        offsets=[-1, 0, 1],
        format="csr"
    )

    # Known solution and RHS
    x_true = np.ones(n, dtype=np.float64)
    b = A @ x_true

    # Transfer CSR components to GPU
    setup_start = time.perf_counter()

    values = cuda.to_device(A.data.astype(np.float64))
    col_indices = cuda.to_device(A.indices.astype(np.int32))
    row_ptr = cuda.to_device(A.indptr.astype(np.int32))

    cuda.synchronize()

    setup_time = (time.perf_counter() - setup_start) * 1000

    # Execute existing verified CG solver
    max_iter = 2 * n

    cuda.synchronize()
    start = time.perf_counter()

    x_gpu, iterations, residual = gpu_conjugate_gradient(
        values,
        col_indices,
        row_ptr,
        b,
        tol=TOL,
        max_iter=max_iter
    )

    cuda.synchronize()
    solver_time = (time.perf_counter() - start) * 1000

    # Validate result
    x_gpu = np.asarray(x_gpu)

    true_residual = (
        np.linalg.norm(A @ x_gpu - b) / np.linalg.norm(b)
    )

    solution_error = np.linalg.norm(x_gpu - x_true)

    converged = bool(
        np.isfinite(true_residual) and true_residual <= TOL
    )

    return {
        "Matrix Size": n,
        "NNZ": A.nnz,
        "Iterations": iterations,
        "Setup + Transfer (ms)": round(setup_time, 3),
        "Solver Time (ms)": round(solver_time, 3),
        "True Relative Residual": true_residual,
        "Solution Error": solution_error,
        "Converged": converged
    }


# -----------------------------
# Run complete benchmark
# -----------------------------

def run_all_benchmarks(progress=gr.Progress()):

    results = []

    for i, n in enumerate(MATRIX_SIZES):

        progress(
            (i + 1) / len(MATRIX_SIZES),
            desc=f"Running CG for N={n}"
        )

        try:
            result = execute_cg(n)
            results.append(result)

        except Exception as e:
            raise gr.Error(
                f"Benchmark failed for N={n}: {str(e)}"
            )

    df = pd.DataFrame(results)

    # Save results
    df.to_csv(RESULT_FILE, index=False)

    return df, RESULT_FILE


# -----------------------------
# Gradio application
# -----------------------------

with gr.Blocks(
    title="Cloud Conjugate Gradient Application"
) as app:

    gr.Markdown("""
    # Cloud-Based Conjugate Gradient Solver

    ### GPU Computing Project – Week 9

    **Platform:** Google Colab Free

    **GPU:** NVIDIA Tesla T4

    This application executes the Conjugate Gradient algorithm
    on a cloud GPU and validates the numerical solution.
    """)

    gr.Markdown("""
    ### Benchmark Configuration

    - Algorithm: Conjugate Gradient
    - Matrix: Sparse tridiagonal SPD
    - Precision: float64
    - Stopping tolerance: 1e-6
    - GPU programming: Numba CUDA
    """)

    run_button = gr.Button(
        "Run All CG Benchmarks",
        variant="primary"
    )

    results_table = gr.Dataframe(
        label="Cloud GPU Benchmark Results",
        interactive=False
    )

    download_file = gr.File(
        label="Download Benchmark CSV"
    )

    run_button.click(
        fn=run_all_benchmarks,
        inputs=None,
        outputs=[results_table, download_file]
    )

    gr.Markdown("""
    ---
    **Project:** Performance and Scalability Analysis
    of the Conjugate Gradient Method

    **Cloud Environment:** Google Colab Free
    """)

# Launch application
app.launch(share=True)


Cloud GPU: Tesla T4
CUDA available: True
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb5bc03c843675fe97.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
